[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sergiovillanueva/Modelos_Fundacionales/blob/main/6_Otras_Tareas.ipynb)

# Otras Tareas de Visión: Pose, OCR, Superresolución, Depth y más

Este notebook cubre aplicaciones especializadas de visión por computador usando modelos fundacionales. Son ejemplos prácticos para que veas las posibilidades y te inspires para tus propios proyectos.

**Tareas que veremos:**
- **Estimación de Pose**: Detectar keypoints del cuerpo humano
- **OCR**: Extraer texto de imágenes
- **Superresolución**: Mejorar calidad de imágenes
- **Background Removal**: Eliminar fondos
- **Depth Estimation**: Calcular profundidad

Todos usando modelos de Hugging Face, listos para usar sin entrenar.

## Configuración e Imports

In [ ]:
import torch
import transformers
from transformers import AutoProcessor, RTDetrForObjectDetection, VitPoseForPoseEstimation
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from transformers import Swin2SRForImageSuperResolution, AutoImageProcessor as Swin2SRProcessor
from transformers import pipeline
from PIL import Image, ImageDraw
import numpy as np
import matplotlib.pyplot as plt
import cv2
import requests
from io import BytesIO
import warnings
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version: {torch.__version__}, usando: {device}")
print(f"transformers version: {transformers.__version__}")

---
## PARTE 1: Estimación de Pose Humana

La estimación de pose detecta puntos clave (keypoints) del cuerpo humano: hombros, codos, muñecas, caderas, rodillas, tobillos, etc.

**Aplicaciones:**
- Análisis deportivo y corrección de técnica
- Rehabilitación y fisioterapia  
- Seguridad laboral (detección de posturas peligrosas)
- Videojuegos y realidad aumentada

**Proceso:**
1. **Detectar personas** con RT-DETR
2. **Detectar keypoints** con VitPose (17 puntos COCO)
3. **Visualizar skeleton** y analizar movimientos

In [ ]:
# Cargar modelos para detección de personas y pose
print("Cargando modelos de pose...")
person_processor = AutoProcessor.from_pretrained("PekingU/rtdetr_r50vd_coco_o365")
person_model = RTDetrForObjectDetection.from_pretrained("PekingU/rtdetr_r50vd_coco_o365").to(device)

pose_processor = AutoProcessor.from_pretrained("usyd-community/vitpose-plus-base")
pose_model = VitPoseForPoseEstimation.from_pretrained("usyd-community/vitpose-plus-base").to(device)

print("✓ Modelos de pose cargados")

In [ ]:
def detect_pose(image, person_threshold=0.3, keypoint_threshold=0.3):
    """Detección completa de pose: personas + keypoints + visualización"""
    
    # Paso 1: Detectar personas
    inputs = person_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = person_model(**inputs)
    
    results = person_processor.post_process_object_detection(
        outputs, target_sizes=torch.tensor([(image.height, image.width)]), threshold=person_threshold
    )
    
    person_boxes = results[0]["boxes"][results[0]["labels"] == 0].cpu().numpy()
    if len(person_boxes) == 0:
        print("No se detectaron personas")
        return None
    
    # Convertir a formato COCO (x, y, w, h)
    person_boxes[:, 2] = person_boxes[:, 2] - person_boxes[:, 0]
    person_boxes[:, 3] = person_boxes[:, 3] - person_boxes[:, 1]
    
    # Paso 2: Detectar keypoints
    inputs_pose = pose_processor(image, boxes=[person_boxes], return_tensors="pt").to(device)
    inputs_pose["dataset_index"] = torch.tensor([0], device=device)
    
    with torch.no_grad():
        outputs_pose = pose_model(**inputs_pose)
    
    pose_results = pose_processor.post_process_pose_estimation(
        outputs_pose, boxes=[person_boxes], threshold=keypoint_threshold
    )
    
    # Paso 3: Visualizar
    img_array = np.array(image).copy()
    
    # Skeleton connections (COCO format)
    skeleton = [
        (15, 13), (13, 11), (16, 14), (14, 12), (11, 12),
        (5, 11), (6, 12), (5, 6), (5, 7), (6, 8), (7, 9), (8, 10),
        (0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (4, 6)
    ]
    
    for person_pose in pose_results[0]:
        kpts_dict = {}
        for keypoint, label, score in zip(person_pose["keypoints"], person_pose["labels"], person_pose["scores"]):
            if score.item() > keypoint_threshold:
                kpts_dict[label.item()] = (int(keypoint[0].item()), int(keypoint[1].item()))
        
        # Dibujar skeleton
        for pt1_idx, pt2_idx in skeleton:
            if pt1_idx in kpts_dict and pt2_idx in kpts_dict:
                cv2.line(img_array, kpts_dict[pt1_idx], kpts_dict[pt2_idx], (0, 255, 0), 3)
        
        # Dibujar keypoints
        for kpt in kpts_dict.values():
            cv2.circle(img_array, kpt, 5, (255, 0, 0), -1)
    
    # Mostrar resultados
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    axes[1].imshow(img_array)
    axes[1].set_title(f"Pose Detection ({len(person_boxes)} personas)")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    return pose_results[0]

print("✓ Función detect_pose() lista")

In [ ]:
# Ejemplo: Detectar pose en imagen de COCO
url = "http://images.cocodataset.org/val2017/000000000139.jpg"
image_pose = Image.open(requests.get(url, stream=True).raw)

pose_results = detect_pose(image_pose)

**Nota sobre análisis de movimientos**: Los keypoints se pueden usar para calcular ángulos de articulaciones (codo, rodilla, etc.), útil para análisis deportivo o fisioterapia. Por ejemplo:

```python
def calculate_angle(p1, p2, p3):
    """Calcula ángulo entre tres puntos (p2 es el vértice)"""
    a = np.array(p1); b = np.array(p2); c = np.array(p3)
    ba = a - b; bc = c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

# Ángulo del codo derecho (hombro-codo-muñeca)
angle = calculate_angle(shoulder, elbow, wrist)
```

---
## PARTE 2: OCR - Reconocimiento de Texto

Extraer texto de imágenes es útil para digitalizar documentos, leer etiquetas industriales, procesar facturas, etc.

Usaremos **TrOCR**, un modelo de Microsoft que combina Vision Transformer (ViT) con GPT para leer texto.

In [ ]:
# Cargar modelo TrOCR
ocr_processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
ocr_model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed").to(device)

print("✓ Modelo OCR cargado")

In [ ]:
def extract_text(image):
    """Extrae texto de imagen usando TrOCR"""
    
    # Procesar imagen
    pixel_values = ocr_processor(image, return_tensors="pt").pixel_values.to(device)
    
    # Generar texto
    with torch.no_grad():
        generated_ids = ocr_model.generate(pixel_values)
    
    text = ocr_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    print(f"Texto extraído: {text}")
    
    # Visualizar
    plt.imshow(image)
    plt.title(f"Texto: {text}")
    plt.axis("off")
    plt.show()
    
    return text

# Ejemplo: imagen con texto simple
# Nota: TrOCR funciona mejor con imágenes de texto recortadas (una línea)
img_text = Image.new('RGB', (300, 60), color='white')
draw = ImageDraw.Draw(img_text)
draw.text((10, 20), "Hello World 2025", fill='black')

text = extract_text(img_text)

**Nota sobre OCR**: Para OCR más robusto en imágenes complejas, considera usar **EasyOCR** o **PaddleOCR** que detectan y leen múltiples líneas automáticamente.

---
## PARTE 3: Superresolución - Mejora de Calidad

La superresolución aumenta la resolución de imágenes de baja calidad. Útil para:
- Mejorar imágenes antiguas o comprimidas
- Zoom digital sin pérdida de calidad
- Restauración de imágenes

Usaremos **Swin2SR**, un modelo basado en Swin Transformer.

In [ ]:
# Cargar modelo Swin2SR (x2 upscaling)
sr_processor = Swin2SRProcessor.from_pretrained("caidas/swin2SR-classical-sr-x2-64")
sr_model = Swin2SRForImageSuperResolution.from_pretrained("caidas/swin2SR-classical-sr-x2-64").to(device)

print("✓ Modelo Superresolución cargado")

In [ ]:
def upscale_image(image):
    """Aumenta resolución de imagen usando Swin2SR"""
    
    # Procesar imagen
    inputs = sr_processor(image, return_tensors="pt").to(device)
    
    # Aplicar superresolución
    with torch.no_grad():
        outputs = sr_model(**inputs)
    
    # Post-procesar
    output = outputs.reconstruction.squeeze().cpu().clamp(0, 1).numpy()
    output = np.transpose(output, (1, 2, 0))  # CHW -> HWC
    output = (output * 255).astype(np.uint8)
    
    output_img = Image.fromarray(output)
    
    # Visualizar comparación
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].imshow(image)
    axes[0].set_title(f"Original {image.size[0]}x{image.size[1]}")
    axes[0].axis("off")
    
    axes[1].imshow(output_img)
    axes[1].set_title(f"Upscaled {output_img.size[0]}x{output_img.size[1]}")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    return output_img

# Ejemplo: cargar imagen de baja resolución
url_img = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/fruits.jpg"
img_low_res = Image.open(BytesIO(requests.get(url_img).content)).convert("RGB")

# Reducir resolución primero para demostrar
w, h = img_low_res.size
img_low_res = img_low_res.resize((w//2, h//2), Image.BILINEAR)

# Aplicar superresolución
img_high_res = upscale_image(img_low_res)

---
## PARTE 4: Background Removal - Eliminación de Fondo

Eliminar el fondo de imágenes es útil para:
- Ecommerce: aislar productos para catálogos
- Fotografía: cambiar fondos
- Videos: efectos especiales

Usaremos **RMBG (Remove Background)** de BRIA AI.

In [ ]:
# Cargar pipeline de background removal
# Este modelo ejecuta codigo del propio repositorio (trust_remote_code), que es lo primero
# que se rompe cuando cambian las librerias. Por eso avisamos en vez de fallar sin mas.
try:
    bg_pipe = pipeline(
        task="image-segmentation",
        model="briaai/RMBG-1.4",
        device=device,
        trust_remote_code=True
    )
    print("✓ Modelo Background Removal cargado")
except Exception as error:
    bg_pipe = None
    print("No se pudo cargar RMBG-1.4:", error)
    print("Alternativa: prueba con briaai/RMBG-2.0 o salta esta parte.")

In [ ]:
def remove_background(image):
    """Elimina fondo de imagen"""
    
    # Aplicar modelo (devuelve imagen RGBA sin fondo)
    no_bg_image = bg_pipe(image)
    
    # Poner sobre fondo negro
    black_bg = Image.new('RGB', no_bg_image.size, (0, 0, 0))
    black_bg.paste(no_bg_image, (0, 0), no_bg_image)
    
    # Mostrar
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    axes[1].imshow(black_bg)
    axes[1].set_title("Sin Fondo")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()

# Ejemplo
url_person = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/person_cars.jpg"
img_person = Image.open(BytesIO(requests.get(url_person).content)).convert("RGB")

remove_background(img_person)

---
## PARTE 5: Depth Estimation - Estimación de Profundidad

Calcular la profundidad de cada píxel en una imagen permite:
- Reconstrucción 3D
- Navegación de robots
- Realidad aumentada
- Efectos de desenfoque selectivo

Usaremos **Depth Anything V2**, un modelo SOTA.

In [ ]:
# Cargar pipeline de depth estimation
depth_estimator = pipeline(
    "depth-estimation",
    model="depth-anything/Depth-Anything-V2-Small-hf",
    device=0 if torch.cuda.is_available() else -1
)

print("✓ Modelo Depth Estimation cargado")

In [ ]:
def estimate_depth(image):
    """Estima mapa de profundidad de imagen"""
    
    # Aplicar depth estimation
    result = depth_estimator(image)
    
    depth_map = result['depth']
    depth_array = np.array(depth_map)
    
    # Normalizar para visualización
    depth_normalized = (depth_array - depth_array.min()) / (depth_array.max() - depth_array.min())
    
    # Visualizar con diferentes colormaps
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    axes[0, 0].imshow(image)
    axes[0, 0].set_title("Original")
    axes[0, 0].axis("off")
    
    im1 = axes[0, 1].imshow(depth_normalized, cmap='viridis')
    axes[0, 1].set_title("Depth Map (Viridis)")
    axes[0, 1].axis("off")
    plt.colorbar(im1, ax=axes[0, 1], label="Profundidad")
    
    im2 = axes[1, 0].imshow(depth_normalized, cmap='plasma')
    axes[1, 0].set_title("Depth Map (Plasma)")
    axes[1, 0].axis("off")
    plt.colorbar(im2, ax=axes[1, 0], label="Profundidad")
    
    im3 = axes[1, 1].imshow(depth_normalized, cmap='jet')
    axes[1, 1].set_title("Depth Map (Jet)")
    axes[1, 1].axis("off")
    plt.colorbar(im3, ax=axes[1, 1], label="Profundidad")
    
    plt.tight_layout()
    plt.show()
    
    return depth_normalized

# Ejemplo: estimar profundidad
url_scene = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/cars.jpg"
img_scene = Image.open(BytesIO(requests.get(url_scene).content)).convert("RGB")

depth = estimate_depth(img_scene)

---
## Ejercicio: Experimenta con tus imágenes

Elige una de las tareas que hemos visto y pruébala con tus propias imágenes:

1. **Pose**: Detecta pose en una foto deportiva
2. **OCR**: Saca foto de un texto impreso
3. **Superresolución**: Mejora una foto antigua de baja calidad
4. **Background Removal**: Aíslate del fondo en una foto tuya
5. **Depth**: Visualiza la profundidad de una escena

### Pruebalo contigo mismo

Haz una foto con la camara y pasale la pose o la profundidad. Ponte de pie y separado de la camara para que se vea el cuerpo entero.


In [ ]:
# Hacer una foto con la camara del portatil (solo funciona en Colab)
import sys
from base64 import b64decode

def hacer_foto(nombre="foto.jpg", calidad=0.92):
    """Abre la camara, espera a que pulses el boton y devuelve la ruta de la foto."""
    if "google.colab" not in sys.modules:
        raise RuntimeError("Esta celda solo funciona en Google Colab. En local, carga un fichero con Image.open().")

    from IPython.display import display, Javascript
    from google.colab.output import eval_js

    display(Javascript("""
        async function tomarFoto(calidad) {
          const div = document.createElement('div');
          const boton = document.createElement('button');
          boton.textContent = 'Hacer foto';
          boton.style.margin = '8px';
          const video = document.createElement('video');
          video.style.display = 'block';
          video.style.maxWidth = '480px';
          const stream = await navigator.mediaDevices.getUserMedia({video: true});
          document.body.appendChild(div);
          div.appendChild(boton);
          div.appendChild(video);
          video.srcObject = stream;
          await video.play();
          google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
          await new Promise((resolve) => boton.onclick = resolve);
          const canvas = document.createElement('canvas');
          canvas.width = video.videoWidth;
          canvas.height = video.videoHeight;
          canvas.getContext('2d').drawImage(video, 0, 0);
          stream.getVideoTracks()[0].stop();
          div.remove();
          return canvas.toDataURL('image/jpeg', calidad);
        }
        """))
    datos = eval_js("tomarFoto({})".format(calidad))
    with open(nombre, "wb") as salida:
        salida.write(b64decode(datos.split(",")[1]))
    return nombre


In [ ]:
ruta = hacer_foto()
mi_foto = Image.open(ruta).convert("RGB")

detect_pose(mi_foto)
estimate_depth(mi_foto)


In [ ]:
# EJERCICIO: Prueba con tus imágenes

# Carga tu imagen
# tu_imagen = Image.open("ruta/a/tu/imagen.jpg").convert("RGB")

# Elige una tarea:
# detect_pose(tu_imagen)
# extract_text(tu_imagen)
# upscale_image(tu_imagen)
# remove_background(tu_imagen)
# estimate_depth(tu_imagen)

## Ejercicio Extra: Combina dos tareas

Crea un pipeline que combine dos de estas tareas. Por ejemplo:
- Eliminar fondo + superponer en imagen con depth
- OCR + clasificar tipo de documento
- Superresolución + eliminar fondo

¡Sé creativo!

In [ ]:
# EJERCICIO EXTRA: Pipeline combinado
def my_custom_pipeline(image):
    """Tu pipeline personalizado"""
    # Tu código aquí...
    pass

# Prueba:
# my_custom_pipeline(img_person)

---
## Más Tareas para Explorar

Hay muchas más tareas que puedes explorar en Hugging Face:

**Vision:**
- Image-to-Image Translation (estilo, día a noche, etc.)
- Video Classification
- Video Segmentation
- Zero-Shot Image Classification
- Unconditional Image Generation (generar imágenes desde cero)

**Vision + Texto:**
- Visual Question Answering (VQA)
- Document Question Answering
- Image Captioning
- Text-to-Image (Stable Diffusion, DALL-E)

**Especializadas:**
- Medical Imaging (rayos X, resonancias)
- Satellite Imagery (análisis de satélites)
- Industrial Inspection (defectos, anomalías)

**Explora**: https://huggingface.co/tasks

---
## Resumen del Curso Completo

A lo largo de estos notebooks hemos visto:

### Viernes:
✅ **Notebook 1**: Detección de objetos con RF-DETR  
✅ **Notebook 2**: Introducción a Hugging Face Hub  
✅ **Notebook 3**: Modelos multimodales (CLIP, BLIP, Grounding DINO, Qwen2.5-VL)  

### Sábado:
✅ **Notebook 4**: DINO v3 y aprendizaje auto-supervisado  
✅ **Notebook 5**: SAM2 y pipelines de segmentación  
✅ **Notebook 6**: Otras tareas (Pose, OCR, SR, Background Removal, Depth)  

### Conceptos Clave:
- Modelos fundacionales: entrenados en datos masivos, aplicables a múltiples tareas
- Zero-shot learning: usar modelos sin reentrenar
- Multimodalidad: combinar visión y lenguaje
- Pipelines: combinar modelos para tareas complejas
- Transformers: arquitectura dominante en IA moderna

### Donde Seguir:
- **Hugging Face Course**: https://huggingface.co/learn
- **Papers with Code**: https://paperswithcode.com
- **Awesome Computer Vision**: https://github.com/jbhuang0604/awesome-computer-vision

**Contacto**: Si tienes dudas puedes escribirme un email!

**¡Gracias por participar en el curso!** 🚀